# 01 — Exploratory Data Analysis
**Dataset:** European Credit Card Fraud (ULB), 284,807 transactions over 2 days (Sept 2013), 492 frauds (0.172%).

**Phase 1 deliverable** — this notebook audits the data (missing values, duplicates, class balance) and explores the patterns that motivate the dissertation's two core challenges: **class imbalance** and **concept drift**. Findings feed Chapter 3 (Methodology) and Chapter 5 (Results).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from fraud import config, data

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

## 1. Load and audit

In [ ]:
df_raw = data.load_raw()
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Missing values and duplicates
n_missing = df_raw.isna().sum().sum()
n_dup = df_raw.duplicated().sum()
print(f"Missing values: {n_missing}")
print(f"Exact duplicate rows: {n_dup}")

df = data.clean(df_raw)
print(f"After cleaning: {df.shape} ({len(df_raw) - len(df)} duplicates removed)")

## 2. Class imbalance — the first core challenge

In [ ]:
counts = df["Class"].value_counts()
fraud_rate = df["Class"].mean()
print(f"Legitimate: {counts[0]:,}")
print(f"Fraud:      {counts[1]:,}")
print(f"Fraud rate: {fraud_rate:.4%}  (roughly 1 fraud per {int(1/fraud_rate):,} transactions)")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot.bar(ax=ax, color=["#4878d0", "#d65f5f"])
ax.set_yscale("log")
ax.set_xticklabels(["Legitimate (0)", "Fraud (1)"], rotation=0)
ax.set_ylabel("Count (log scale)")
ax.set_title("Class distribution — note the log scale")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(config.FIGURES_DIR / "eda_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

A classifier that predicts "legitimate" for **every** transaction achieves ~99.83% accuracy while catching zero fraud. This is why accuracy (and to a lesser degree ROC-AUC) is misleading here, and why this project reports precision, recall, F1, **PR-AUC** and **MCC** instead (Baisholan et al., 2025).

## 3. Transaction amount by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, name, color in [(0, "Legitimate", "#4878d0"), (1, "Fraud", "#d65f5f")]:
    axes[0].hist(df.loc[df["Class"] == label, "Amount"], bins=80,
                 alpha=0.6, label=name, color=color, density=True)
axes[0].set_xlim(0, 500)
axes[0].set_xlabel("Amount (EUR)")
axes[0].set_ylabel("Density")
axes[0].set_title("Amount distribution (zoom < 500)")
axes[0].legend()

sns.boxplot(data=df, x="Class", y=np.log1p(df["Amount"]), hue="Class",
            ax=axes[1], palette=["#4878d0", "#d65f5f"], legend=False)
axes[1].set_xticklabels(["Legitimate", "Fraud"])
axes[1].set_ylabel("log(1 + Amount)")
axes[1].set_title("Log-amount by class")

fig.savefig(config.FIGURES_DIR / "eda_amount_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

df.groupby("Class")["Amount"].describe().round(2)

The literature reports that fraud concentrates at low amounts (mostly < EUR 500; Albalawi & Dardouri, 2025) — small charges are less likely to be noticed by the cardholder. The summary table above lets us confirm this on our copy of the data.

## 4. Time patterns — motivation for the concept-drift analysis

In [ ]:
# Time = seconds elapsed since the first transaction; the data covers ~48h.
df["hour"] = (df["Time"] // 3600) % 24

hourly = df.groupby("hour").agg(
    n_transactions=("Class", "size"),
    n_fraud=("Class", "sum"),
)
hourly["fraud_rate"] = hourly["n_fraud"] / hourly["n_transactions"]

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].bar(hourly.index, hourly["n_transactions"], color="#4878d0")
axes[0].set_ylabel("Transactions")
axes[0].set_title("Volume and fraud rate by hour of day")
axes[1].bar(hourly.index, hourly["fraud_rate"] * 100, color="#d65f5f")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].set_xlabel("Hour of day")
fig.savefig(config.FIGURES_DIR / "eda_hourly_pattern.png", dpi=150, bbox_inches="tight")
plt.show()

Fraud rate spikes in the small hours when legitimate volume is low — consistent with Afriyie et al. (2023), who found peak fraud between 22:00 and 04:00. The transaction distribution is clearly **not stationary over time**, which is exactly why Phase 4 evaluates models with chronological splits instead of random ones.

In [ ]:
# How does the fraud rate differ between the two days?
df["day"] = (df["Time"] // 86400).astype(int)
df.groupby("day").agg(
    n_transactions=("Class", "size"),
    n_fraud=("Class", "sum"),
    fraud_rate=("Class", "mean"),
).style.format({"fraud_rate": "{:.4%}"})

## 5. Which features discriminate fraud?

In [ ]:
v_cols = [f"V{i}" for i in range(1, 29)]
corr = df[v_cols + ["Amount"]].corrwith(df["Class"]).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
corr.plot.barh(ax=ax, color=np.where(corr > 0, "#d65f5f", "#4878d0"))
ax.set_xlabel("Pearson correlation with Class")
ax.set_title("Feature correlation with fraud label")
fig.savefig(config.FIGURES_DIR / "eda_feature_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

print("Strongest (absolute) correlations:")
print(corr.abs().sort_values(ascending=False).head(10).round(3))

In [ ]:
# Distributions of the top discriminative features
top4 = corr.abs().sort_values(ascending=False).head(4).index.tolist()
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, top4):
    for label, name, color in [(0, "Legit", "#4878d0"), (1, "Fraud", "#d65f5f")]:
        ax.hist(df.loc[df["Class"] == label, col], bins=60, alpha=0.6,
                label=name, color=color, density=True)
    ax.set_title(col)
    ax.legend()
fig.suptitle("Top discriminative features: class-conditional distributions")
fig.savefig(config.FIGURES_DIR / "eda_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

Ali (SSRN working paper) reports V4, V14, V10 and V12 among the most important SHAP features — the correlation ranking above gives us an early, model-free view to compare against the Phase 5 SHAP analysis. Because V1–V28 are PCA-anonymised, business interpretation of individual features is impossible; this is an acknowledged limitation (see the portfolio's Ethical Issues section).

## 6. EDA findings (summary for the dissertation)

1. **Data quality:** no missing values; ~1,081 exact duplicate rows removed to prevent train/test contamination.
2. **Extreme imbalance:** fraud rate ≈ 0.17% — accuracy is a meaningless headline metric; PR-AUC/MCC required.
3. **Amount:** fraudulent transactions concentrate at low amounts, consistent with the literature.
4. **Time:** fraud rate varies strongly by hour (peaks overnight) and between the two days — the distribution is non-stationary, motivating chronological validation (Phase 4).
5. **Features:** V14, V17, V12, V10 (and neighbours) show the strongest class separation — a reference point for the SHAP analysis in Phase 5.
